# Yorùbá OCR Research — Google Colab Pipeline

**Runtime:** GPU (T4 or better) → Runtime → Change runtime type → T4 GPU

> **Data:** Keep `data/` on Drive under the repo folder (`My Drive/yoruba_ocr_research/data/`). Code tracks GitHub; work in the Drive repo root so paths match `data/processed/`.

**Pipeline (shell phases):**
1. Mount Drive & repo
2. Install deps (Paddle GPU + `requirements.txt`; for **PaddleOCR-VL-1.5** also `pip install 'transformers>=5' peft` for phases 15–16)
3. Clone **PaddleOCR** (PP-OCRv4 **comparison** train/eval; same crops feed VL export)
4. Phase **01** consolidate (if needed) → **02–03** analyse + config
5. Phase **04** PP-OCRv4 CRNN fine-tune (**classical comparison**)
6. Phases **05–08** Paddle eval, Tesseract, Qwen, **PP-OCRv4 ablations** (long)
7. **Primary supervised model (Table 1):** **14** export JSONL → **15** zero-shot VL (`SKIP_VL15_EVAL=0`) → **16** LoRA → **15** with `--adapter-path` for main metrics — see `docs/vl15_pipeline.md`
8. **Optional:** phases **12** / **13** diagnostics & strict alignment
9. Phase **09** compile tables + `eval_alignment_report.json`
10. Phase **99** backup to Drive

**Not in repo:** Mistral OCR. HF model id: `PaddlePaddle/PaddleOCR-VL-1.5`.

## Step 1 — Mount Drive & Set Up Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
REPO_DIR   = f'{DRIVE_ROOT}/yoruba_ocr_research'
DATA_DIR   = f'{REPO_DIR}/data'

# The data/ folder lives at: My Drive/yoruba_ocr_research/data/
# Upload was done manually because GitHub doesn't support large binary files.
# We work INSIDE the Drive folder — no copying is needed.
print(f'Drive mounted.')
print(f'Repo + data directory : {REPO_DIR}')
print(f'data/processed exists : {os.path.isdir(DATA_DIR + "/processed")}')
print(f'data/raw exists       : {os.path.isdir(DATA_DIR + "/raw")}')

In [ ]:
import os

if not os.path.isdir(REPO_DIR + '/.git'):
    # First time: clone code from GitHub into the Drive folder.
    # The data/ subfolder already exists there (uploaded manually), so git
    # will only add the code files on top — it won't touch data/.
    !git clone https://github.com/sam4rano/yoruba_ocr_research.git "{REPO_DIR}"
    print('Repo cloned.')
else:
    # Returning session: just pull the latest code changes.
    print('Repo already exists — pulling latest code...')
    !git -C "{REPO_DIR}" pull --ff-only

# Set working directory to repo root (data/ is already here)
%cd "{REPO_DIR}"
!pwd
!ls data/

## Step 2 — Install Python Dependencies

In [ ]:
# Install PaddlePaddle GPU (Colab Linux x86 with CUDA)
!python -m pip install paddlepaddle-gpu -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q

In [ ]:
# Install all other project requirements (skip paddlepaddle line — already installed above)
!grep -v '^paddlepaddle' requirements.txt | grep -v '^#' | grep -v '^$' > /tmp/reqs_no_paddle.txt
!python -m pip install -r /tmp/reqs_no_paddle.txt -q
print('All dependencies installed.')

In [ ]:
# Verify paddle is importable and sees the GPU
import paddle
print('PaddlePaddle version:', paddle.__version__)
print('GPU available:', paddle.device.is_compiled_with_cuda())
print('Device count :', paddle.device.cuda.device_count())

## Step 3 — Clone PaddleOCR Repo

In [ ]:
import os

if not os.path.isdir('PaddleOCR'):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git PaddleOCR
else:
    print('PaddleOCR already cloned — skipping.')

# PaddleOCR repo requirements (detection/recognition training)
!python -m pip install -q -r PaddleOCR/requirements.txt
# Qwen 2.5 VL baselines
!python -m pip install -q accelerate qwen-vl-utils bitsandbytes
# PaddleOCR-VL-1.5 (HF) — phases 15–16; use transformers v5 per model card
!python -m pip install -q 'transformers>=5.0.0' peft
print('PaddleOCR repo + VL/Qwen extras installed.')


## Step 4 — Verify Data & Run Analysis + Config (Phases 02-03)

In [ ]:
import os, subprocess

# First, dynamically generate data/processed/ from data/raw/.
# This completely resolves the formats for PaddleOCR, Tesseract, and Qwen.
!bash scripts/shell/phase_01_consolidate.sh

# Now verify the newly built folders are perfectly complete.
checks = {
    # PaddleOCR Data
    'data/processed/labels/train.txt': 'file',
    'data/processed/labels/val.txt':   'file',
    'data/processed/labels/test.txt':  'file',
    'data/processed/dictionary/yoruba_char_dict.txt': 'file',
    'data/processed/images/train':     'dir',
    'data/processed/images/val':       'dir',
    'data/processed/images/test':      'dir',
    # Tesseract Data
    'data/tesseract_finetune/train':   'dir',
    'data/tesseract_finetune/val':     'dir',
    'data/tesseract_finetune/test':    'dir',
    # Qwen Data
    'data/qwen_finetune/train.jsonl':  'file',
    'data/qwen_finetune/val.jsonl':    'file',
    'data/qwen_finetune/test.jsonl':   'file',
}

ok = True
for path, kind in checks.items():
    exists = os.path.isfile(path) if kind == 'file' else os.path.isdir(path)
    if exists:
        if kind == 'dir':
            n = len(os.listdir(path))
            print(f'  ✅  {path}  ({n} items)')
        else:
            n = int(subprocess.check_output(['wc', '-l', path]).split()[0])
            print(f'  ✅  {path}  ({n} lines)')
    else:
        print(f'  ❌  MISSING: {path}')
        ok = False

if not ok:
    raise RuntimeError('data/processed is incomplete. Ensure data/raw/ has the correct exports.')

print('\n✅  All data checks passed — ready to train!')


In [ ]:
# Phase 02 — Dataset analysis (re-runs fast, confirms dict coverage)
!bash scripts/shell/phase_02_analyze.sh

# Phase 03 — Generate PaddleOCR YAML config explicitly setting GPU mode
!CONFIG_FORCE_GPU=1 bash scripts/shell/phase_03_config.sh

print('Phases 02 and 03 complete.')


In [ ]:
# Sanity check: show key fields from the generated config
import yaml
with open('configs/paddleocr_yoruba_rec.yml') as f:
    cfg = yaml.safe_load(f)

g = cfg.get('Global', {})
print('use_gpu  :', g.get('use_gpu'))
print('char_dict:', g.get('character_dict_path'))
print('epochs   :', g.get('epoch_num'))
print('save_dir :', g.get('save_model_dir'))

## Step 5 — Fine-Tune PaddleOCR on GPU (Phase 04)

In [ ]:
import os
os.environ['EVAL_USE_GPU']     = '1'
os.environ['CONFIG_FORCE_GPU'] = '1'

# Set TRAIN_RESUME=1 to automatically find the latest checkpoint
# in experiments/finetuned/ and resume training from there.
os.environ['TRAIN_RESUME'] = '1'

# Optional overrides:
# os.environ['TRAIN_EPOCHS_OVERRIDE'] = '100'
# os.environ['TRAIN_BATCH_OVERRIDE'] = '64'

!bash scripts/shell/phase_04_train.sh

## Step 6 — Evaluate baselines (Phases 05–08)

Phases 05 (Paddle), 06 (Tesseract), 07 (Qwen), 08 (ablations). Optional **PaddleOCR-VL-1.5** steps are in the next cell.

In [ ]:
# Phase 05: Evaluate PaddleOCR (pretrained baseline + fine-tuned checkpoint)
!EVAL_USE_GPU=1 bash scripts/shell/phase_05_eval_paddle.sh

In [ ]:
# Install Tesseract system binary + Yoruba language pack
!apt-get install -qq tesseract-ocr tesseract-ocr-yor 2>/dev/null

# Phase 06: Tesseract baseline
!bash scripts/shell/phase_06_tesseract.sh

In [ ]:
# Phase 07: Qwen 2.5-VL zero-shot baseline
# Needs ~16GB VRAM. 4-bit quantize is enabled by default in the script.
import os
os.environ['SKIP_QWEN'] = '0'  # Explicitly enable Qwen (skipped by default otherwise)
!bash scripts/shell/phase_07_qwen.sh


In [ ]:
# Phase 08: Ablation study
!bash scripts/shell/phase_08_ablation.sh

### Primary supervised — PaddleOCR-VL-1.5 (HF)

This is the **main fine-tuned system** for Yorùbá line crops (LoRA on **16**, eval on **15**). PP-OCRv4 phase **04** is the classical CRNN comparison, not the primary supervised row in Table 1.

- **14** exports JSONL under `data/paddleocr_vl15_sft/` (does not alter `data/processed/`).
- **15** zero-shot VL — set `SKIP_VL15_EVAL=0` (GPU); baseline before LoRA.
- **16** LoRA training — long GPU job; then **15** with `--adapter-path experiments/paddleocr_vl15_lora/adapter` for `paddleocr_vl15_lora_finetuned`.

**Optional QA:** **12** hypothesis checks; **13** strict alignment: `VERIFY_STRICT=1 bash scripts/shell/phase_13_verify_eval.sh`.

See `docs/vl15_pipeline.md`.

In [ ]:
# Phase 14 — VL-1.5 SFT export (CPU-safe)
!bash scripts/shell/phase_14_export_vl15.sh

# Phase 15 — VL-1.5 eval (uncomment; needs GPU + HF cache)
# import os
# os.environ['SKIP_VL15_EVAL'] = '0'
# !bash scripts/shell/phase_15_eval_vl15.sh

# Phase 16 — LoRA fine-tune (uncomment; very long)
# !bash scripts/shell/phase_16_train_vl15_lora.sh
# Then: fine-tuned metrics
# !python scripts/15_baseline_paddleocr_vl15.py --adapter-path experiments/paddleocr_vl15_lora/adapter

# Optional sanity: bash scripts/shell/phase_12_diagnose.sh

## Step 8 — Compile results (Phase 09)

In [ ]:
!bash scripts/shell/phase_09_compile.sh

# Table 1: metrics_summary.csv == table1_main_comparison.csv
# eval_alignment_report.json — non-empty "mismatches" => re-run evals on current data
import json
from pathlib import Path
import pandas as pd

df = pd.read_csv('results/tables/metrics_summary.csv')
print(df.to_string(index=False))
rep = Path('results/tables/eval_alignment_report.json')
if rep.is_file():
    r = json.loads(rep.read_text(encoding='utf-8'))
    if r.get('mismatches'):
        print('\n[align] Mismatch: stored n != current label counts — see', rep)
    else:
        print('\n[align] OK —', rep)

## Step 9 — Backup results to Google Drive (Phase 99)

In [ ]:
import os
# Results + model checkpoints go to a timestamped subfolder on your Drive.
os.environ['DRIVE_BACKUP_ROOT']    = f'{DRIVE_ROOT}/yoruba_ocr_backups'
os.environ['BACKUP_EXPERIMENTS']   = '1'  # also backup model checkpoints

!bash scripts/shell/phase_99_backup.sh
print('Backup complete. Check: My Drive/yoruba_ocr_backups/')